# 🧭 生成式AI應用開發｜第04週｜150分鐘修訂版
## Prompt Engineering：從訊息結構到可驗證的小功能
延續第三周 Responses API，以固定 few-shot 範例銜接第五周對話紀錄。
本檔另存自原實作教材，原版保留。先備：list/dict、for、函式、instructions、role/content。

> **學生版**：訊息組裝、客服分類器與擇一應用保留 TODO；先完成練習再開啟付費區塊。

## 🎯 學習目標
1. 區分工作身分與訊息 role，使用 instructions 搭配訊息列表。
2. 用角色、任務、背景、限制與輸出格式寫清楚規格。
3. 用迴圈建立 user/assistant few-shot 範例，再附上本次 user 輸入。
4. 用同一批案例比較 zero-shot、few-shot 與邊界範例版本。
5. 完成合法標籤檢查，並完成改寫或依據資料問答其中一項。
6. 分清 API 失敗、非法格式與答案錯誤，依證據記錄修訂。

## 🗺️ 三堂課：150分鐘有效授課時間（休息另計）
| 分鐘 | 活動 | 產出 |
|---|---|---|
| 0–15 | 環境與第三周複習 | 區分規則、資料與訊息角色 |
| 15–35 | 基準比較、五元素 | 改良 prompt 與觀察紀錄 |
| 35–55 | instructions＋訊息列表、分隔符、任務拆解 | 讀懂並修改請求 |
| 55–80 | zero-shot／few-shot、訊息組裝必做 | 固定範例列表與對照 |
| 80–110 | 客服分類器與五題測試必做 | v1/v2/v3 結果與修正 |
| 110–135 | 改寫／資料問答擇一 | 檢查輸出與失敗案例 |
| 135–150 | 同儕覆核、版本紀錄、下周銜接 | 修訂理由與完成檢核 |
摘要是共同示範；分類是核心實作；改寫／問答擇一，未選題課後完成。
教師課前先完成環境準備；若 API 排障超過 5 分鐘，先做離線訊息與規格檢查。

---

## 1️⃣ 環境準備

沿用第 3 週設定：API key 放在 Colab Secrets 的 `OPENAI_API_KEY`，不要直接寫進 Notebook。程式會先檢查金鑰，再建立 API client；若設定不完整，會提供明確錯誤訊息。


In [ ]:
# 安裝或更新 OpenAI Python SDK
!pip install -q --upgrade openai


In [ ]:
import os
from openai import OpenAI

# Colab 優先讀取 Secrets；本機執行時則使用作業系統環境變數。
try:
    from google.colab import userdata
    secret_key = userdata.get("OPENAI_API_KEY")
    if secret_key:
        os.environ["OPENAI_API_KEY"] = secret_key
except ImportError:
    # 非 Colab 環境時，沿用作業系統環境變數。
    pass
except Exception as exc:
    print(f"無法讀取 Colab Secret：{exc}")


# 沒有金鑰也能完成訊息列表與規格的離線練習；付費入口會明確提示。
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
client = OpenAI() if os.getenv("OPENAI_API_KEY") else None
RUN_DEMOS = False
print("本週模型：", MODEL)
print("API 已可用" if client else "離線練習模式：尚未設定 API key")

In [ ]:
def require_text(value, name):
    """在本機先拒絕空白或錯誤型別，避免無效的付費請求。"""
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"{name} 必須是非空白字串。")


def request_text(api_input, instructions):
    """共用 API 與錯誤處理；各入口先驗證自己的輸入結構。"""
    require_text(instructions, "instructions")
    if client is None:
        raise RuntimeError("請先設定 OPENAI_API_KEY 並重新執行環境 cell。")
    try:
        response = client.responses.create(
            model=MODEL, instructions=instructions, input=api_input, store=False,
        )
    except Exception as exc:
        # API 失敗與模型答錯分開處理，不把錯誤文字當成正常答案。
        raise RuntimeError(f"OpenAI API 呼叫失敗：{exc}") from exc
    if getattr(response, "status", "completed") != "completed":
        raise RuntimeError("回應尚未完成，不採用部分文字。")
    for item in getattr(response, "output", []) or []:
        for part in getattr(item, "content", []) or []:
            if getattr(part, "type", None) == "refusal":
                raise RuntimeError("模型拒絕此請求，請檢查資料與任務。")
    result = response.output_text
    if not isinstance(result, str) or not result.strip():
        raise RuntimeError("模型沒有回傳可用文字。")
    return result


def ask_ai(user_input, instructions="你是一位清楚、可靠的繁體中文助理"):
    """沿用單次文字任務入口，摘要、改寫與資料問答使用此函式。"""
    require_text(user_input, "user_input")
    return request_text(user_input, instructions)


def ask_ai_messages(messages, instructions):
    """接收有角色的訊息列表；本週只使用 user/assistant 固定範例。"""
    if not isinstance(messages, list) or not messages:
        raise ValueError("messages 必須是非空列表。")
    for message in messages:
        if not isinstance(message, dict) or message.get("role") not in {"user", "assistant"}:
            raise ValueError("本週訊息 role 必須是 user 或 assistant。")
        require_text(message.get("content"), "content")
    if messages[-1]["role"] != "user":
        raise ValueError("最後一則必須是本次待回答的 user 訊息。")
    return request_text(messages, instructions)


def compare_prompts(user_input, prompt_variants):
    """同一筆文字搭配不同 instructions；每個版本各付費一次。"""
    if not isinstance(prompt_variants, dict) or not prompt_variants:
        raise ValueError("prompt_variants 必須是非空字典。")
    return {name: ask_ai(user_input, rules) for name, rules in prompt_variants.items()}

> **成本提醒**：付費示範與版本測試預設關閉。`RUN_DEMOS=True` 的共同示範共 10 次請求；同 cell 重跑仍會付費。
> 版本測試另需 3×5＝15 次；擇一應用改寫 1 次或問答 3 次。先完成練習再分段開啟，不必全部重跑。
> 記錄模型、規則版本、範例與案例；一次結果不代表穩定改善。`store=False` 不代表資料沒有傳送給 OpenAI。

---

## 2️⃣ 從模糊 prompt 建立基準結果

先保留簡單版本作為 baseline。之後每次只調整一個主要變因，才能判斷是哪項改動造成差異。


In [ ]:
course_intro = """
生成式AI應用開發是一門大三課程，學生會使用 Python、OpenAI API、
Streamlit 與 GitHub 完成 AI 應用。課程重視實作、API key 安全、成本控制，
並以期中個人專題和期末小組專題整合所學。
"""

# 用 XML-like 標籤包住資料，可降低模型把指令與資料混在一起的機率。
comparison_input = f"""請摘要以下課程資料：
<course>
{course_intro}
</course>"""
# baseline prompt 故意保持簡短，稍後用來和明確版本比較差異。
baseline_instructions = "請摘要使用者提供的課程資料。"
if RUN_DEMOS:
    baseline_result = ask_ai(comparison_input, baseline_instructions)
    print(baseline_result)

### ✍️ 觀察紀錄

- 摘要有幾句？是否使用繁體中文？
- 是否保留課程工具與評量方式？
- 重跑一次時，長度與格式是否相同？

模糊 prompt 不一定失敗，但它沒有先定義「什麼叫成功」。


---

## 3️⃣ Prompt 的五個實用元素

| 元素 | 要回答的問題 | 範例 |
|---|---|---|
| 角色 Role | 模型以什麼身份工作？ | 大學課程助教 |
| 任務 Task | 具體完成什麼？ | 摘要課程介紹 |
| 背景 Context | 必要資料是什麼？ | `<course>...</course>` |
| 限制 Constraints | 長度、語言、禁忌？ | 60 字內、不新增資訊 |
| 輸出 Output | 結果長什麼樣？ | 一句總結＋三個重點 |

口訣：**誰、做什麼、根據什麼、遵守什麼、輸出什麼。**


In [ ]:
# 明確版本把角色、資料限制、輸出長度與重點數都寫進 instructions。
clear_instructions = """
# Identity
你是大學課程的教學助理，擅長整理繁體中文課程資訊。

# Instructions
1. 只能根據使用者提供的課程文字摘要，不得新增資訊。
2. 先寫一句 60 字以內的總結。
3. 再列三個重點，涵蓋技術、課程特色與評量。
4. 使用繁體中文。
"""
if RUN_DEMOS:
    clear_result = ask_ai(comparison_input, clear_instructions)
    print(clear_result)
# 已保留兩份輸出，不另重複呼叫 compare_prompts；它可供課後新版本比較。

### ✍️ 預測、觀察、修正（兩版皆填）
執行前先圈出五元素；預測明確版本可能改善哪一項，再比較實際結果。
| 面向 | 基準實際結果 | 明確版實際結果 | 通過條件 |
|---|---|---|---|
| 格式 |  |  | 一句總結＋三點 |
| 長度 |  |  | 總結 60 字元內 |
| 必要資訊 |  |  | 技術、特色、評量 |
| 事實 |  |  | 沒有新增原文未提供的資訊 |
不預填「一定改善」；結果可能相同或退步。下一版只調整一個主要變因。
字數以 Python 字元數計，包含標點；換行可排除，但不等於 token。

---

## 4️⃣ `instructions` 與 `input`：規則和資料分開

- `instructions`：應用程式提供的固定角色、規則、流程與範例。
- `input`：每次請求會改變的使用者任務與資料。

課堂常把固定高層規則泛稱為 system prompt；本課使用 Responses API，因此用 `instructions` 實作。可把固定規則想成函式定義，把每次輸入想成函式參數。


### 🔗 接回第三周：工作身分與訊息 role
| 名稱 | 意義 | 本週放法 |
|---|---|---|
| 工作身分 | 助教、摘要器、分類器 | instructions 的角色描述 |
| developer | 開發者提供的高層規則 | 第三周示範；本週規則統一放 instructions |
| user | 範例輸入或本次任務 | input 列表中的訊息 |
| assistant | 提供給模型的範例答案／先前回答 | 本週用人工提供的示範答案 |
`input` 可以是字串，也可以是訊息列表。`instructions` 和訊息列表可以一起使用。
本週的 assistant 範例是事先寫好的，沒有在此新增一次 API 呼叫，也不是訓練模型。
第五周才累積真正的 user/assistant 歷史；每次只送相同固定範例不會自動記住上一輪。

In [ ]:
# 先印出請求結構；只開啟 RUN_DEMOS 才真的送出一次請求。
bridge_messages = [{"role": "user", "content": "請用一句話解釋 API。"}]
print(bridge_messages)
if RUN_DEMOS:
    print(ask_ai_messages(bridge_messages, "你是資管系助教，使用繁體中文與生活化例子。"))

In [ ]:
def summarize_text(text, max_chars_per_point=35, num_points=3):
    # 將可調整的輸出限制設成參數，讓同一個函式能重複用在不同資料。
    if max_chars_per_point <= 0 or num_points <= 0:
        raise ValueError("字數與重點數必須是正整數。")

    instructions = f"""
# Identity
你是精準的繁體中文摘要助理。
# Instructions
- 只根據提供的文字作答，不新增內容。
- 保留重要名稱、數字與限制。
- 輸出一個標題與 {num_points} 個條列重點。
- 每點不超過 {max_chars_per_point} 字。
"""
    # 原文放在 input；摘要規則放在 instructions，維持 prompt 結構清楚。
    user_input = f"""請摘要：
<source_text>
{text}
</source_text>"""
    return ask_ai(user_input, instructions)



if RUN_DEMOS:
    print(summarize_text(course_intro, max_chars_per_point=35, num_points=3))

### 🧱 Markdown 與 XML 分隔資料

- Markdown 標題讓 prompt 層次清楚、容易維護。
- XML 標籤指出資料與範例的開始、結束。
- 分隔符能降低規則與資料混淆，但不是完整安全機制；外部文字仍是不可信資料。


---

## 5️⃣ 任務拆解：把複雜要求排成步驟

當任務包含多個判斷，明確寫出可觀察的工作流程，通常比只寫「仔細思考」更容易測試。


In [ ]:
feedback = "介面很漂亮，但註冊流程一直失敗，客服兩天後才回覆。"
# Workflow 先要求模型按步驟思考，Output format 再約束最後輸出欄位。
analysis_instructions = """
# Identity
你是產品回饋分析助理。
# Workflow
1. 找出稱讚項目。2. 找出問題。3. 判斷最優先問題。4. 提出一項改善建議。
# Output format
- 優點：
- 問題：
- 優先處理：
- 建議：
"""

if RUN_DEMOS:
    print(ask_ai(f"<feedback>{feedback}</feedback>", analysis_instructions))

### 小任務：先讀規格再執行
請把任務拆解的四步對應到四個輸出欄位；自行指出一項原文無法支持的推測。
分隔符提升可讀性，不能保證防止提示注入；資料中的指令仍應視為資料。

---

## 6️⃣ Zero-shot 與 Few-shot

- **Zero-shot**：描述規則，不提供答案範例。
- **Few-shot**：提供少量且具代表性的輸入／輸出範例，讓模型掌握標籤、格式與邊界。

範例應涵蓋多樣情況；太相似的範例無法清楚呈現分類界線。


### Few-shot 的兩種呈現方式
原版以 `<example><input>…</input><output>…</output></example>` 寫進高層指令，這也是有效方式。
本版主要用 `user`／`assistant` 訊息配對，練習第三周的 list/dict 與第五周會用到的訊息結構。
兩種方式沒有保證孰優；本次實驗固定用訊息列表，只改有無範例，避免同時改兩個變因。

### 必做小練習：組裝訊息（先離線）
四組 `(輸入, 標籤)` 應變成 8 則示範訊息，再加 1 則待分類訊息。
依序確認：user→assistant 配對、最後是 user、content 是文字；最後輸入不附答案。

In [ ]:
def build_few_shot_messages(text, examples):
    """練習：將範例配對轉成訊息列表；此步驟不呼叫 API。"""
    require_text(text, "text")
    # TODO 1：建立空 messages 列表。
    # TODO 2：用 for 解開每個 (sample_input, sample_output)，附上 user/assistant 配對。
    # TODO 3：最後附上本次 text 的 user 訊息，回傳 messages。
    # 提示：四組範例會產生 8 則示範訊息，再加 1 則本次輸入。
    raise NotImplementedError("請完成訊息組裝練習，再開啟相關示範。")

In [ ]:
COURSE_CATEGORIES = ["內容", "進度", "設備", "其他"]
COURSE_EXAMPLES = [
    ("希望 API 範例多一點。", "內容"),
    ("操作速度太快。", "進度"),
    ("教室電腦無法安裝套件。", "設備"),
    ("希望下次分組討論。", "其他"),
]
COURSE_RULES = """你是課程意見分類器，只輸出內容、進度、設備、其他其中一個標籤。
教材主題與範例歸內容；速度與練習時間歸進度；網路、電腦、投影可讀性歸設備；其餘歸其他。
若同時提到設備與進度，優先歸設備；其餘多類衝突依設備、進度、內容、其他的順序選一類。
"""


def classify_course(text, examples):
    # zero-shot 與 few-shot 共用規則、請求結構與輸出驗證，只改變範例數。
    messages = build_few_shot_messages(text, examples)
    result = ask_ai_messages(messages, COURSE_RULES).strip()
    if result not in COURSE_CATEGORIES:
        raise ValueError(f"非法分類：{result!r}")
    return result


def course_zero_shot(text):
    return classify_course(text, [])


def classify_feedback(text):
    return classify_course(text, COURSE_EXAMPLES)


if RUN_DEMOS:
    probe = "投影看不清楚，而且老師切換畫面太快。"
    print("Zero-shot：", course_zero_shot(probe))
    print("Few-shot：", classify_feedback(probe))

In [ ]:
RUN_MESSAGE_CHECK = False
if RUN_MESSAGE_CHECK:
    # 此檢查不呼叫 API，完成組裝函式後可先開啟。
    messages = build_few_shot_messages("練習時間不夠。", COURSE_EXAMPLES)
    assert len(messages) == 9
    assert [m["role"] for m in messages] == ["user", "assistant"] * 4 + ["user"]
    assert messages[-1]["content"] == "練習時間不夠。"
    print(messages)

### ✍️ Few-shot 觀察與成功條件
比較題「投影看不清楚，而且老師切換畫面太快」同時涉及設備、進度。
先閱讀規則：本例優先選設備，才能訂出可評估的預期答案。
| 版本 | 預期 | 實際 | 有無額外文字 | 是否改善 |
|---|---|---|---|---|
| zero-shot | 設備 |  |  |  |
| few-shot | 設備 |  |  |  |
請說明 assistant 範例的來源，以及它和真正對話紀錄的差別。

<div style="border-left:5px solid #f9ab00;padding:10px 14px;background:#fff8e1;">
<font color="#b06000"><b>先不要混淆</b></font>：本週只用文字規則限制輸出，仍可能出現意外格式；第 6 週再用 Structured Outputs 與 schema 做程式層級保證。
</div>


## 🛠️ 核心必做：客服分類器與版本測試（80–110分）
定義四類，並採單一標籤。多類衝突依「帳務問題→功能問題→操作疑問→其他」優先序。
v1：只有規則；v2：相同規則＋每類一例；v3：相同規則＋v2 範例＋一筆衝突範例。
這個優先序是本練習的產品規格，可以討論，但必須先決定再訂測試答案。
先寫預期答案，再執行五題固定測試；不要為了通過而把測試題抄入範例。

In [ ]:
SUPPORT_LABELS = ["功能問題", "帳務問題", "操作疑問", "其他"]
# 範例與測試集分開：不要把後面的測試題直接放進範例。
SUPPORT_TESTS = [
    {"input": "我的信用卡被扣款兩次。", "expected": "帳務問題"},
    {"input": "我要在哪裡修改登入密碼？", "expected": "操作疑問"},
    {"input": "按儲存後出現錯誤，資料消失了。", "expected": "功能問題"},
    {"input": "謝謝客服快速回覆！", "expected": "其他"},
    {"input": "扣款兩次，而且報表頁面打不開。", "expected": "帳務問題"},
]

# TODO 1：寫出身分、四類定義、衝突優先序與只輸出標籤的限制。
SUPPORT_RULES_V1 = ""
# TODO 2：每類各寫一筆代表性範例，避免複製測試題。
SUPPORT_EXAMPLES_V2 = []
# TODO 3：保留 v2 範例並增加一筆符合優先規則的衝突範例。
SUPPORT_EXAMPLES_V3 = []


def classify_support_message(message, examples):
    # TODO 4：呼叫 build_few_shot_messages 與 ask_ai_messages，去除首尾空白。
    # TODO 5：檢查輸出屬於 SUPPORT_LABELS，否則 raise ValueError，再回傳標籤。
    raise NotImplementedError("請完成客服分類器。")


def support_v1(text):
    return classify_support_message(text, [])


def support_v2(text):
    return classify_support_message(text, SUPPORT_EXAMPLES_V2)


def support_v3(text):
    return classify_support_message(text, SUPPORT_EXAMPLES_V3)

### 測試方式：函式作為參數
`run_classification_tests(..., support_v1, "v1")` 傳入的是函式，沒有加括號。
測試器逐筆執行 `classify_fn(case["input"])` 才呼叫它；不需要 lambda。
下方提供測試器，學生把重點放在規則、範例與失敗原因。

In [ ]:
def run_classification_tests(test_cases, classify_fn, version):
    """接受分類函式，讓同一批案例能測不同版本；保留錯誤類別便於判讀。"""
    rows = []
    for case in test_cases:
        try:
            actual = classify_fn(case["input"])
            passed = actual == case["expected"]
            status = "通過" if passed else "模型答案錯誤"
        except NotImplementedError:
            # 尚未完成練習時明確中止，不把空白答案當作分類失敗。
            raise
        except RuntimeError as exc:
            actual, passed, status = str(exc), False, "API／回應失敗"
        except ValueError as exc:
            actual, passed, status = str(exc), False, "輸入／輸出契約錯誤"
        row = dict(version=version, input=case["input"], expected=case["expected"],
                   actual=actual, passed=passed, status=status, model=MODEL)
        rows.append(row)
        print(row)
    print(f"{version}：{sum(row['passed'] for row in rows)}/{len(rows)}")
    return rows


RUN_VERSION_TESTS = False
version_results = {}
if RUN_VERSION_TESTS:
    # 每版 5 題，共 15 次請求；完成 TODO 並確認額度後才開啟。
    for version, classify_fn in {"v1": support_v1, "v2": support_v2, "v3": support_v3}.items():
        version_results[version] = run_classification_tests(SUPPORT_TESTS, classify_fn, version)

### 🧪 Prompt 改版紀錄（五題固定案例）
| 版本 | 修改 | 通過 | API／回應失敗 | 契約錯誤 | 保留／退回與原因 |
|---|---|---|---|---|---|
| v1 | 明確規則，無範例 | /5 |  |  |  |
| v2 | 加入四類範例 | /5 |  |  |  |
| v3 | 增加衝突範例 | /5 |  |  |  |
少量測試只代表這批案例，不足以證明整體穩定性。API 失敗不應直接解讀為 prompt 退步。
若全數通過，請另加一筆未看過的挑戰題；若失敗，先檢查規格是否明確，再修改一個變因。

## 🔀 四種應用與擇一實作（110–135分）
摘要已共同示範，分類已必做。下方保留改寫與依據資料問答的示範函式。
課堂選 B 或 C 一題實作並驗證，另一題課後完成；不要把全部示範與兩題都當成課堂必做。

### C. 改寫：指定讀者、語氣與不可改變的事實


In [ ]:
def rewrite_notice(text, audience="大學生", tone="友善", max_chars=100):
    # audience、tone、max_chars 參數化後，同一個改寫函式可套用不同情境。
    if max_chars <= 0:
        raise ValueError("max_chars 必須是正整數。")

    instructions = f"""
# Identity
你是校園行政文字編輯。
# Instructions
- 改寫成適合{audience}閱讀的{tone}繁體中文通知。
- 不得更改日期、時間、地點與應繳項目。
- 輸出一個短標題及一段通知，總字數不超過 {max_chars} 字。
"""
    result = ask_ai(f"<original_notice>{text}</original_notice>", instructions)
    # 模型可能超出字數限制，因此仍要在程式端做第二層檢查。
    if len(result.replace("\n", "")) > max_chars:
        print(f"⚠️ 字數檢查：模型輸出超過 {max_chars} 字，請調整 prompt 或改用結構化流程。")
    return result


notice = "本課程作業二應於10月15日23:59前上傳Moodle，逾期系統關閉，不收email補交。"

if RUN_DEMOS:
    print(rewrite_notice(notice, audience="大學生", tone="友善", max_chars=100))

### D. 依據資料問答：限定可使用的 context

這是後續 RAG 的基礎。模型只能根據提供資料回答；資料沒有答案時，必須清楚回報資料不足。


In [ ]:
def answer_from_context(
    question,
    context,
    missing_reply="資料不足，請詢問授課教師。",
):
    # 這是 RAG 問答的雛形：答案只能來自 context，找不到就回固定文字。
    instructions = f"""
# Identity
你是課程資訊問答助理。
# Instructions
- 只能根據 `<course_info>` 回答，不可用常識補充規定。
- 無法回答時固定回覆「{missing_reply}」
- 可以回答時先給直接答案，再附簡短依據。
"""
    # context 與 question 分開包標籤，讓模型知道哪段是資料、哪段是問題。
    data = f"<course_info>{context}</course_info><question>{question}</question>"
    return ask_ai(data, instructions)


course_rules = "作業二截止時間為10月15日23:59，繳交平台為Moodle。"

if RUN_DEMOS:
    print(answer_from_context("作業二要交到哪裡？", course_rules))
    print(answer_from_context("作業二占學期成績幾分？", course_rules))

### 擇一 B：學習內容改寫器
保留專有名詞並白話解釋、加入生活化比喻、最後一題理解題不給答案，250 字元內。
先預測應保留哪些事實，再檢查專有名詞、標題、字元數與有無新增無根據細節。

In [ ]:
# 練習 B：用 template 保留 prompt 結構，只把 max_chars 當成變數代入。
BEGINNER_REWRITE_PROMPT_TEMPLATE = """
# TODO 1：加入 Identity、Instructions、{max_chars} 參數與 Output format
"""


def rewrite_for_beginner(text, max_chars=250):
    # 提示：先檢查 max_chars，再用 template.format(max_chars=max_chars) 產生 instructions。
    # TODO 2：檢查 max_chars 必須大於 0
    # TODO 3：用 format() 代入字數，用 <technical_text> 包住資料並呼叫 ask_ai
    # TODO 4：若輸出超過 max_chars，印出警告
    # 提示：prompt 限制不是保證，程式仍需做字數檢查。
    raise NotImplementedError("請完成改寫器。")


technical_text = """
Embedding 是把文字轉換成高維度數值向量的技術。語意相近的文字，
其向量在空間中的距離通常較近，因此可以用來進行語意搜尋。
"""

RUN_PRACTICE_B = False
if RUN_PRACTICE_B:
    # TODO 5：完成函式後執行，保存結果供後面的本機檢查使用。
    raise NotImplementedError("請將本段改成呼叫並保存 beginner_result。")

In [ ]:
def check_beginner_output(result, max_chars=250):
    # 這些是可確定的格式檢查；比喻、事實與是否給出理解題答案需人工閱讀。
    return {
        "字元數符合": len(result.replace("\n", "")) <= max_chars,
        "保留專有名詞": "Embedding" in result,
        "必要標題": "## 白話說明" in result and "## 想一想" in result,
    }

if RUN_PRACTICE_B and "beginner_result" in globals():
    print(check_beginner_output(beginner_result))

### 擇一 C：依據課程規定問答
測試「能回答」、「與資料衝突的前提」、「資料未提供」三種問題。
第三題必須固定回覆資料不足；前兩題則人工對照 policy，不要求逐字相同。
政策資料是本練習 fixture，不用它推定其他周的正式專題規定。

In [ ]:
# 練習 C：policy 是唯一可用資料來源，回答不可使用常識補充。
policy = """
期中專題採個人製作，於第8週課堂展示。作品需包含簡單操作介面、
至少一次 API 呼叫與錯誤處理。展示時間每人4分鐘。
"""
questions = ["期中專題可以兩人一組嗎？", "每人展示多久？", "期中專題占總成績幾分？"]


def answer_policy_question(
    question,
    missing_reply="資料不足，請詢問授課教師。",
):
    # TODO 1：建立 instructions，只能依 policy 回答，並代入 missing_reply
    # 提示：明確要求「找不到答案時固定回覆 missing_reply」。
    # TODO 2：組合 policy 與 question，再呼叫 ask_ai
    # 提示：用 <policy> 與 <question> 分隔資料和問題。
    raise NotImplementedError("請完成資料問答。")


RUN_PRACTICE_C = False
if RUN_PRACTICE_C:
    # TODO 3：逐題呼叫並將回答依序保存到 policy_answers，供本機檢查使用。
    raise NotImplementedError("請完成問題迴圈。")

In [ ]:
if RUN_PRACTICE_C and "policy_answers" in globals():
    # 資料不足可做精確檢查；有答案的事實正確性仍需人工核對。
    assert len(policy_answers) == 3
    assert policy_answers[2].strip() == "資料不足，請詢問授課教師。"
    print("資料不足固定回覆檢查通過；請人工確認個人製作與每人4分鐘。")

### ✍️ 擇一應用觀察紀錄
| 測試輸入／問題 | 成功條件 | 實際結果 | 程式檢查 | 人工覆核／修正 |
|---|---|---|---|---|
| 一般案例 |  |  |  |  |
| 邊界／資料不足 |  |  |  |  |
程式只檢查部分契約；字數或標題通過不代表事實正確。

### 同儕檢查（135–150分）
交換規則與測試紀錄：對方能否不用猜測就判斷衝突標籤？範例是否誤把測試答案洩漏？
指出一個失敗或仍未驗證的條件，完成一次修改並寫下原因。

## ✅ 完成檢核
- [ ] 能區分工作身分與 API role，讀懂 instructions＋input 訊息列表。
- [ ] 完成離線 few-shot 組裝，正確附上本次問題。
- [ ] 寫出五元素，能說明分隔符的作用與限制。
- [ ] 完成客服分類器，檢查合法標籤與衝突優先序。
- [ ] 用五題相同案例比較三版本並區分失敗類型。
- [ ] 完成改寫或資料問答，至少一項程式檢查及人工覆核。
- [ ] 說明固定範例與對話歷史的差別；本週格式要求仍不是 Structured Outputs。

## 📝 課後任務與後續銜接
繳交訊息組裝結果、客服 v1/v2/v3 規則與範例、五題比較表（另加一題挑戰）、
擇一應用輸出與檢查、模型名稱與150字修訂反思；未選題作延伸練習。
不提交 API key、真實個資、成績或其他敏感資料。
第五周：把固定範例的 user/assistant 結構延伸成真正對話 history，學習記憶、reset 與 Streaming。
第六周：使用 Structured Outputs 控制 schema；本週的 prompt 與程式檢查保留作為基礎。

## 📚 延伸閱讀

- OpenAI Prompt engineering：https://developers.openai.com/api/docs/guides/prompt-engineering
- OpenAI Text generation：https://developers.openai.com/api/docs/guides/text

下週將進入：**多輪對話、conversation state 與 Streaming 輸出**。
